In [1]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt

In [2]:
def get_run_dir(parent_dir, run_id):
    run_name = str(run_id) if str(run_id).startswith('run') else f'run{run_id}'
    return parent_dir / run_name


def load_accuracy(metrics_path):
    if not metrics_path.exists():
        raise FileNotFoundError(f'Metrics file not found: {metrics_path}')
    with metrics_path.open(newline='') as handle:
        rows = list(csv.DictReader(handle))
    if not rows:
        raise ValueError(f'Metrics file is empty: {metrics_path}')
    required = {'round', 'test_accuracy'}
    missing = required - set(rows[0])
    if missing:
        raise ValueError(f'Missing columns {missing} in {metrics_path}')
    rounds = [int(row['round']) for row in rows]
    accuracy = [100.0 * float(row['test_accuracy']) for row in rows]
    return rounds, accuracy


def plot_accuracy_comparison(baseline_dir, noniid_dir, baseline_run_id, noniid_run_id):
    baseline_metrics = get_run_dir(Path(baseline_dir), baseline_run_id) / 'metrics.csv'
    noniid_metrics = get_run_dir(Path(noniid_dir), noniid_run_id) / 'metrics.csv'
    baseline_rounds, baseline_accuracy = load_accuracy(baseline_metrics)
    noniid_rounds, noniid_accuracy = load_accuracy(noniid_metrics)

    plt.figure(figsize=(10, 6))
    plt.plot(baseline_rounds, baseline_accuracy, marker='o', markersize=3, label=f'IID baseline (run {baseline_run_id})')
    plt.plot(noniid_rounds, noniid_accuracy, marker='s', markersize=3, label=f'Non-IID (run {noniid_run_id})')
    plt.xlabel('FedAvg round')
    plt.ylabel('Test accuracy (%)')
    plt.title('IID baseline vs non-IID FedAvg')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f'IID final accuracy: {baseline_accuracy[-1]:.2f}%')
    print(f'Non-IID final accuracy: {noniid_accuracy[-1]:.2f}%')
    return None

In [3]:
# Change the run IDs and directories here before running this cell.
BASELINE_RUN_ID = 9
NONIID_RUN_ID = 11

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; using local fallback paths.')

PROJECT_ROOT = Path('/content/drive/MyDrive/FLasIRL(new)')
BASELINE_DIR = PROJECT_ROOT / 'baseline'
NONIID_DIR = PROJECT_ROOT / 'fedavg_noniid'

plot_accuracy_comparison(BASELINE_DIR, NONIID_DIR, BASELINE_RUN_ID, NONIID_RUN_ID)

Mounted at /content/drive


FileNotFoundError: Metrics file not found: /content/drive/MyDrive/FLasIRL_new/baseline/run9/metrics.csv